In [ ]:
import psycopg2, random
import pandas as pd

random.seed(1234)

## Notebook Purpose
<b> This is notebook number 1 </b>

This notebook looks into the postgres tables public.Image, public.Tags, and a few others to gather training data for all models in the mixture including the NLP transformers, the ViT transformers, and the traditional CV models

### Notebook Order
1. getData
2. downloadData
3. trainResNetModel | trainPromptTransformerClassifier | trainViTClassifier
4. localMixtureEval

In [ ]:
url = #REMOVED TO PROTECT PII

conn = psycopg2.connect(url)
cur = conn.cursor()

# Query to get all table names
cur.execute("SELECT table_name FROM information_schema.tables WHERE table_schema='public'")
tables = cur.fetchall()
conn = psycopg2.connect(url)

Get data users have voted on at least 3 times with 50%+ agreement

In [ ]:
moderator_images = pd.read_sql_query(sql_query, conn)


In [ ]:
moderator_images.head()

In [ ]:
moderator_images['mod_nsfw_level'] = moderator_images['mod_rating'].apply(lambda x: 'PG' if x == 1 else 'PG13' if x == 2 else 'R' if x == 4 else 'X' if x == 8 else 'XXX' if x == 16 else 'Blocked')
moderator_images = moderator_images[~moderator_images['mod_nsfw_level'].isin(['Blocked'])]

moderator_images.groupby('mod_nsfw_level')['download_url'].count()
image_with_ids = moderator_images 

In [ ]:
# image_with_ids.groupby('reviewed_level')['download_url'].count()

Reviewed level data  (training)
| label | count (4/05/24)| count (5/13/24)
| - | - | - |
| PG |3287| 4404 
|PG13| 6060| 9188
|R|  2422| 3483 
|X| 3037| 6540 |
|XXX |2795| 5021 |

Reviewed level data (evaluation)
| label | count (5/22/24)
| - | - | 
| PG | 4604 
|PG13| 4318
|R|  4051 
|X|  3264
|XXX |2306


In [ ]:
image_with_ids.head()

In [ ]:
imageID = image_with_ids['id'].values

In [ ]:
query = f"""
SELECT t."imageId", array_agg(t."tagId") as tags
FROM public."TagsOnImage" t
WHERE "imageId" in ({','.join([str(i) for i in imageID])})
AND t."needsReview" IS FALSE
and t.disabled IS FALSE
and t.source != 'Rekognition'
GROUP BY t."imageId"
"""


In [ ]:
tagsInts = pd.read_sql_query(query, conn, params=(imageID,))

# Close the database connection
conn.close()


In [ ]:
image_with_ids = image_with_ids.merge(tagsInts, left_on='id', right_on='imageId', how='inner')

Check data previously used in models

In [ ]:
previous_data = pd.read_csv('../data/13k_community_images_and_seb_moderated_images.csv')
previous_data_all = pd.read_csv('../data/reviewed_data_5_13_24.csv')

In [ ]:
previous_data = pd.concat([previous_data, previous_data_all])

In [ ]:
previous_data.groupby('label')['download_url'].count()

In [ ]:
prev_image_ids = previous_data['id'].values

In [ ]:
new_images = image_with_ids[~image_with_ids['id'].isin(prev_image_ids)]

In [ ]:
new_images.head()

See new/unique image counts

In [ ]:
new_images.groupby('nsfw_level')['download_url'].count()

In [ ]:
image_with_ids = image_with_ids.drop(columns= 'nsfw_level').rename(columns = {"mod_nsfw_level": "label",
                                                                              "tags_x": "tags",
                                                                              "tags_y": "tag_ids"})
image_with_ids.head()

In [ ]:
image_with_ids.groupby('label')['download_url'].count()

In [ ]:
image_with_ids.to_csv('../data/moderated_data_7_30_24.csv', index=False)

get all previous data and concatonate

In [ ]:
reviewed_data_for_training = pd.concat([image_with_ids, previous_data]).drop_duplicates(subset='id')
reviewed_data_for_training.groupby('label')['download_url'].count()

Save the full dataset

In [ ]:
reviewed_data_for_training.to_csv('./data/reviewed_data_5_13_24.csv', index=False)

break down dataframe into classes and save a subset. Recombine, so we have our static training data

In [ ]:
sample_size = reviewed_data_for_training.groupby('label')['download_url'].count().min()
for label in reviewed_data_for_training['label'].unique():
    if label == reviewed_data_for_training['label'].unique()[0]:
        temp = reviewed_data_for_training[reviewed_data_for_training['label'] == label].sample(n = sample_size)
    else:
        temp = pd.concat([temp, reviewed_data_for_training[reviewed_data_for_training['label'] == label].sample(n = sample_size)])

print(len(temp))
temp.groupby('label')['download_url'].count()

In [ ]:
temp.to_csv('./data/reviewed_data_5_13_24_training.csv', index=False)

In [ ]:
holdout_data = reviewed_data_for_training[~reviewed_data_for_training['id'].isin(temp['id'])]
holdout_data.groupby('label')['download_url'].count()

In [ ]:
holdout_data.to_csv('./data/reviewed_data_5_13_24_holdout.csv', index=False)